# Rapidly-exploring Random Tree

Jonas Frei, 20.10.2026

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation, PillowWriter

## Hilfsfunktionen

Funktion um zu Überprüfen, ob ein Knoten mit einem Hindernis kollidiert

In [ ]:
def is_collision(point, obstacles):
    x, y = point
    inside = ((x >= obstacles[:, 0]) & (x <= obstacles[:, 0] + obstacles[:, 2])
              & (y >= obstacles[:, 1]) & (y <= obstacles[:, 1] + obstacles[:, 3]))
    return bool(inside.any())

Funktion um zu Überprüfen, ob eine Verbindung mit einem Hindernis kollidiert

In [ ]:
def is_segment_collision(p1, p2, obstacles):
    numPoints = 100
    x = np.linspace(p1[0], p2[0], numPoints)
    y = np.linspace(p1[1], p2[1], numPoints)
    inside = ((x[:, None] >= obstacles[None, :, 0]) & (x[:, None] <= obstacles[None, :, 0] + obstacles[None, :, 2])
              & (y[:, None] >= obstacles[None, :, 1]) & (y[:, None] <= obstacles[None, :, 1] + obstacles[None, :, 3]))
    return bool(inside.any())

Funktion zum Finden des nächstgelegenen Knotens

In [ ]:
def nearest_node(nodes, point):
    dists = np.linalg.norm(nodes - point, axis=1)
    return int(np.argmin(dists))

## Parameterdefinition

Maximale Anzahl Knoten

In [ ]:
maxNodes = 500

Schrittweite für das Erstellen neuer Knoten

In [ ]:
stepSize = 5

Kartengrösse

In [ ]:
xRange = [0, 100]
yRange = [0, 100]

Abbruchkriterium (Radius um Zielpunkt)

In [ ]:
goalRadius = 5

Startposition und Zielposition

In [ ]:
start = np.array([10., 10.])
goal = np.array([80., 70.])

Hindernisse als Vierecke [x, y, width, height]

In [ ]:
obstacles = np.array([
    [0, 0, 100, 1],
    [0, 0, 1, 100],
    [0, 99, 100, 1],
    [99, 0, 1, 100],
    [0, 80, 30, 1],
    [40, 60, 1, 40],
    [40, 0, 1, 39],
    [50, 80, 50, 1],
    [0, 49, 40, 1],
    [60, 10, 1, 40],
    [60, 60, 1, 20],
    [60, 50, 40, 1],
], dtype=float)

## Visualisierung

Gitterkarte mit Hindernissen

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(xRange)
ax.set_ylim(yRange)
ax.set_aspect('equal')
ax.set_title("Rapidly-exploring Random Tree (RRT)")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.grid(True)

for obstacle in obstacles:
    ax.add_patch(Rectangle((obstacle[0], obstacle[1]), obstacle[2], obstacle[3], facecolor='black'))

Visualisierung der Startposition und Zielposition

In [ ]:
ax.plot(start[0], start[1], 'go', markersize=10, markerfacecolor='g', label="Start")
ax.plot(goal[0], goal[1], 'ro', markersize=10, markerfacecolor='r', label="Ziel")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
fig

## RRT Algorithmus

In [ ]:
nodes = [start.copy()]
edges = []
ax.plot(start[0], start[1], 'ro')
goalReached = False
frames = []

for i in range(maxNodes):
    # Wähle zufälligen Punkt innerhalb der Karte
    sample = np.array([np.random.rand()*(xRange[1] - xRange[0]) + xRange[0],
                        np.random.rand()*(yRange[1] - yRange[0]) + yRange[0]])

    # Finde den nächstgelegenen Knoten
    nearestIdx = nearest_node(np.array(nodes), sample)
    nearestNode = nodes[nearestIdx]

    # Bestimme die Position des neuen Knotens
    direction = (sample - nearestNode)/np.linalg.norm(sample - nearestNode)
    newNode = nearestNode + stepSize*direction

    # Überprüfe, dass der neue Knoten oder die Verbindung nicht mit einem Hindernis kollidiert
    if not is_collision(newNode, obstacles) and not is_segment_collision(nearestNode, newNode, obstacles):
        nodes.append(newNode)
        edges.append((nearestIdx, len(nodes) - 1))
        ax.plot([nearestNode[0], newNode[0]], [nearestNode[1], newNode[1]], 'b')
        ax.plot(newNode[0], newNode[1], 'ro')
        frames.append(list(edges))

        # Überprüfe, ob der neue Knoten nahe genug an der Zielposition liegt -> Ziel erreicht
        if np.linalg.norm(newNode - goal) <= goalRadius:
            goalReached = True
            break

nodes = np.array(nodes)
fig

## Visualisierung des Pfades

In [ ]:
def find_parent(edges, child_idx):
    for parent, child in edges:
        if child == child_idx:
            return parent
    return None


if goalReached:
    print('Goal reached!')
    path = [len(nodes) - 1]
    while path[-1] != 0:
        path.append(find_parent(edges, path[-1]))

    ax.plot([nodes[path[0], 0], goal[0]], [nodes[path[0], 1], goal[1]], 'c', linewidth=2)
    for i in range(1, len(path)):
        ax.plot([nodes[path[i - 1], 0], nodes[path[i], 0]],
                [nodes[path[i - 1], 1], nodes[path[i], 1]], 'c', linewidth=2)
else:
    print('Goal not reached within the maximum number of nodes.')

fig

## Animation des Baumwachstums

In [ ]:
fig_anim, ax_anim = plt.subplots(figsize=(8, 8))
ax_anim.set_xlim(xRange)
ax_anim.set_ylim(yRange)
ax_anim.set_aspect('equal')
ax_anim.set_title("Rapidly-exploring Random Tree (Animation)")
ax_anim.set_xlabel("X")
ax_anim.set_ylabel("Y")
ax_anim.grid(True)

for obstacle in obstacles:
    ax_anim.add_patch(Rectangle((obstacle[0], obstacle[1]), obstacle[2], obstacle[3], facecolor='black'))

ax_anim.plot(start[0], start[1], 'go', markersize=10, markerfacecolor='g', label="Start")
ax_anim.plot(goal[0], goal[1], 'ro', markersize=10, markerfacecolor='r', label="Ziel")
ax_anim.legend(loc="center left", bbox_to_anchor=(1.3, 0.5))
fig_anim.subplots_adjust(right=0.7)

tree_lines = []
path_lines = []

# Segmente des rekonstruierten Pfades, für die Animation nach dem Baumwachstum
path_segments = []
if goalReached:
    path_segments.append((nodes[path[0]], goal))
    for i in range(1, len(path)):
        path_segments.append((nodes[path[i - 1]], nodes[path[i]]))

holdFrames = 45
totalFrames = len(frames) + len(path_segments) + holdFrames


def update_rrt(frame):
    if frame < len(frames):
        edges_so_far = frames[frame]
        while len(tree_lines) < len(edges_so_far):
            parent_idx, child_idx = edges_so_far[len(tree_lines)]
            line, = ax_anim.plot([nodes[parent_idx, 0], nodes[child_idx, 0]],
                                  [nodes[parent_idx, 1], nodes[child_idx, 1]], 'b')
            tree_lines.append(line)
    else:
        path_frame = min(frame - len(frames) + 1, len(path_segments))
        while len(path_lines) < path_frame:
            p1, p2 = path_segments[len(path_lines)]
            line, = ax_anim.plot([p1[0], p2[0]], [p1[1], p2[1]], 'c', linewidth=2)
            path_lines.append(line)
    return tree_lines + path_lines


anim = FuncAnimation(fig_anim, update_rrt, frames=totalFrames, interval=30, blit=True)
anim.save('rrt_animation.gif', writer=PillowWriter(fps=20))
plt.close(fig_anim)

In [ ]:
from IPython.display import Image

Image(filename='rrt_animation.gif')